# qrepro: reference reproductions

Every published-estimate number this repository claims, recomputed from source and
asserted against the pinned literals in `references/values.py`. Each cell asserts before
it prints, so a dependency bump that moves a number fails the notebook.

| key | paper | scope |
|---|---|---|
| Beverland | [arXiv:2211.07629](https://arxiv.org/abs/2211.07629) | three application instances; Qualtran ships this model, so this checks wiring |
| GE19 | [arXiv:1905.09749v3](https://arxiv.org/abs/1905.09749) | logical reconciliation, the windowed construction, the physical rows |
| G2025 | [arXiv:2505.15917](https://arxiv.org/abs/2505.15917) | decomposed against GE19 through one model, not reproduced |

Sources, free parameters, conventions, tolerances and every named divergence:
[`ASSUMPTIONS.md`](../ASSUMPTIONS.md).

Counting convention: the magic-state currency is `n_ccz = and_bloq + toffoli + cswap`;
one Qualtran `And` = one `Toffoli` = one CCZ = 4 T = one GE19 "Toffoli" (sec. 3).

In [ ]:
import math
import re

from qrepro.references import (
    reproduce_2019_to_2025,
    reproduce_beverland,
    reproduce_ge19_logical,
    reproduce_ge19_physical,
    reproduce_ge19_windowed,
    windowed_total_ccz,
)

# _toffoli_counts applies the same normalisation reproduce_2019_to_2025 applies
# internally, so the counts shown in sec. 5 cannot drift from the rows computed from them.
from qrepro.references.decomposition import CONVENTIONS, _toffoli_counts
from qrepro.references.values import (
    BEVERLAND_TOL,
    G2025,
    G2025_TOL,
    GE19,
    GE19_TOL,
    GE19_WINDOWED,
    GE19_WINDOWED_ACHIEVED,
    GE19_WINDOWED_TOL,
    modexp_toffoli_coset,
    modexp_toffoli_reference,
    modexp_toffoli_windowed,
)

N = GE19["n"]  # 2048
TABLE1 = GE19["table1_toffoli_billions"]
DEFAULT_WINDOW = (GE19_WINDOWED["exp_window"], GE19_WINDOWED["mul_window"])

In [ ]:
# Digits, optional thousands separators, optional exponent, optional unit suffix
# (12.3%, 5.64x, 17.97M). Anything else marks the column as text and left-aligns it.
_NUMERIC = re.compile(r"^[-+]?[\d,]*\.?\d+(?:[eE][-+]?\d+)?[%xM]?$")


def table(headers, rows):
    """Fixed-width table; columns whose cells all parse as numbers right-align."""
    cells = [[f"{v}" for v in row] for row in rows]
    widths = [max([len(h)] + [len(c[i]) for c in cells]) for i, h in enumerate(headers)]
    right = [
        all(_NUMERIC.match(c[i]) for c in cells if c[i]) for i in range(len(headers))
    ]

    def line(values):
        return "  ".join(
            f"{v:{'>' if r else '<'}{w}}" for v, r, w in zip(values, right, widths)
        ).rstrip()

    head = line(headers)
    print(head, "-" * len(head), sep="\n")
    for c in cells:
        print(line(c))


def within(value, target, rel):
    """True if value lies within a relative tolerance of target."""
    return abs(value - target) <= abs(target) * rel


def monotone(values, *, increasing, strict=False):
    """Monotonicity of a sequence. Direction is keyword-only so it cannot be defaulted."""
    pairs = list(zip(values, values[1:]))
    if increasing:
        return all(b > a if strict else b >= a for a, b in pairs)
    return all(b < a if strict else b <= a for a, b in pairs)

## 1. Beverland et al.

Qualtran's Beverland model on the paper's three application instances, against the targets
in `values.py`. Qualtran ships this model, so agreement checks the wiring rather than
showing independent convergence.

Targets are the paper's equations, not its Table I. Chemistry and factoring agree
(sec. V-B L1398, sec. V-C L1422); quantum dynamics does not. Table I prints
$C_{\min} = 1.5 \cdot 10^5$ and $R = 2.4 \cdot 10^6$, both irreconcilable with the paper's
own (D3)/(D4): the measurement count alone ($M_{\text{Meas}} = 1.4 \cdot 10^6$) exceeds the
printed step count 9.3x and the printed T-count is exactly 4x the formula's. (D3)/(D4) at
the paper's $A = 0.53$, $B = 5.3$ give $1.4401 \cdot 10^6$ and $6.02 \cdot 10^5$, which is
what Qualtran's model computes on the inputs as printed and what is targeted here
(ASSUMPTIONS.md sec. 1). Code distances are the paper's printed values throughout,
evaluated at the printed step count rather than the computed $C_{\min}$.

In [ ]:
beverland = reproduce_beverland()

TOL_BY_METRIC = {
    "c_min": BEVERLAND_TOL["rel_c_min"],
    "t_states": BEVERLAND_TOL["rel_t_states"],
    "code_distance": 0.0,  # exact: the paper's own printed distances
}
for r in beverland.rows:
    tol = TOL_BY_METRIC[r.metric]
    assert abs(r.deviation) <= tol, (
        f"{r.label}/{r.metric}: {r.deviation:+.2%} exceeds {tol:.0%}"
    )

table(
    ["instance", "metric", "qrepro", "target", "dev"],
    [
        [
            r.label,
            r.metric,
            f"{r.reproduced:.5g}",
            "" if r.target is None else f"{r.target:.5g}",
            "" if r.deviation is None else f"{r.deviation:+.2%}",
        ]
        for r in beverland.rows
    ],
)

## 2. GE19 logical

GE19's closed form and Qualtran's `ModExp` call-graph count, side by side with the three
modexp regimes GE19 prices. The two differ by ~64x: `ModExp` implements GE19's reference
(non-windowed) construction at `20*n_e*n^2`, not the paper's optimized result. The trailing
line decomposes that ratio into its three published factors.

`QubitCount`, `AlgorithmSummary.from_bloq` and `decompose_bloq` are never called on
`ModExp`, since they trace wires in O(gates) and hang at n >= 128, so the logical-qubit
count comes from GE19's `3n + 0.002*n*lg n`.

In [ ]:
logical = reproduce_ge19_logical()
ne_shor, ne_eh = 2 * N, 1.5 * N

assert (
    GE19_TOL["divergence_lo"] <= logical.divergence_ratio <= GE19_TOL["divergence_hi"]
), f"ModExp/formula divergence {logical.divergence_ratio:.2f}x outside declared band"
assert round(logical.logical_qubits_formula) == GE19["logical_qubits"], (
    f"logical-qubit formula gives {logical.logical_qubits_formula:.1f}, "
    f"expected {GE19['logical_qubits']}"
)

table(
    ["quantity", "value", "target", "dev"],
    [
        [
            "logical qubits (formula)",
            f"{logical.logical_qubits_formula:.1f}",
            GE19["logical_qubits"],
            "",
        ],
        [
            "Toffoli (GE19 formula)",
            f"{logical.toffoli_formula:.4e}",
            f"{GE19['toffoli_count']:.1e}",
            f"{logical.toffoli_formula / GE19['toffoli_count'] - 1:+.2%}",
        ],
        ["Qualtran ModExp (n_ccz)", f"{logical.modexp_ccz_count:,}", "", ""],
        ["ModExp / GE19 formula", f"{logical.divergence_ratio:.2f}x", "", ""],
    ],
)
print()

table(
    ["modexp regime", "source", "n_e", "Toffoli"],
    [
        [
            "reference  20*ne*n^2",
            "S2.2 L522",
            "2n",
            f"{modexp_toffoli_reference(N, ne_shor):.4e}",
        ],
        [
            "coset       8*ne*n^2",
            "S2.4 L547",
            "2n",
            f"{modexp_toffoli_coset(N, ne_shor):.4e}",
        ],
        [
            "windowed 24*ne*n^2/lg^2n",
            "S2.5 L602",
            "1.5n",
            f"{modexp_toffoli_windowed(N, ne_eh):.4e}",
        ],
        ["optimized (Table 1)", "Table 1", "1.5n", f"{GE19['toffoli_count']:.4e}"],
        [
            "MEASURED Qualtran ModExp",
            "call graph",
            "2n",
            f"{logical.modexp_ccz_count:.4e}",
        ],
    ],
)

`n_ccz/(n_e*n^2)` measured across a 64-fold range in n. A constant identifies the
non-windowed reference regime; a windowed construction falls like `1/lg^2 n`, which
section 4 measures.

The constant is half the 20 GE19 derives (sec. 2.2 L522) for the construction Qualtran
documents. The factor of two is the adder primitive: GE19 prices Cuccaro's adder at `2n`
Toffolis, Qualtran's `Add` is Gidney's temporary-AND adder at `n-1` ANDs with the carry
uncomputed by measurement. The closed form printed below is a regression pin, not evidence,
since its coefficient was fitted to the measurement. The evidence is the scaling.

In [ ]:
sizes, coeffs = zip(*logical.coefficient_series)
closed_form = 10 * ne_shor * N**2 + 5 * ne_shor * N
spread = (coeffs[0] - coeffs[-1]) / coeffs[-1]

assert monotone(list(coeffs), increasing=False), (
    "coefficient series is not non-increasing"
)
assert within(coeffs[-1], 10.0, 0.01), (
    f"coefficient {coeffs[-1]:.5f} does not converge to 10"
)
assert spread < 0.02, f"series falls by {spread:.2%}, expected flat"
assert closed_form == logical.modexp_ccz_count, (
    f"closed form {closed_form:,} != call graph {logical.modexp_ccz_count:,}"
)

table(["n", "n_ccz/(ne*n^2)"], [[n, f"{c:.5f}"] for n, c in logical.coefficient_series])

## 3. GE19 windowed construction

`algorithms/windowed_factoring.py` builds what GE19 costs, windowed exponentiation over
windowed multiplication over the coset representation (sec. 2.4-2.5) at Ekera-Hastad
`n_e = 1.5n`, from stock Qualtran leaves:

```
WindowedModExp             ceil(n_e/w_e) uncontrolled multiplications  (L590)
  WindowedModMul           2 multiply-add passes                       (anc:171, L694)
    WindowedMultiplyAdd    ceil((n+g_pad+2)/w_m) lookup additions      (L590)
      LookupAddition
        QROAMClean(log_block_sizes=(0,))   lookup      (L594)
        Add(QUInt(width))                  addition    (L593)
        QROAMClean(...).adjoint()          unlookup    (L595)
```

This is a second derivation of the 2.7e9 regime that does not go through the paper's closed
forms. Every window and padding parameter is GE19-published (`g_exp = g_mul = 5`,
`g_pad = 2 lg n + lg n_e + 10`, L690).

"Bridged" doubles the adder term and only the adder term, converting Qualtran's Gidney
AND-adder to GE19's Cuccaro convention. It is reported beside the unbridged figure, which
is the number this pipeline computes.

`16*ne*n^2/lg^2n` is L602's 24 in Qualtran's adder currency, not a published constant: the
`24 = 2*4*3` has its `3 = 2 adder + 1 lookup` become `2 = 1 + 1`.

In [ ]:
windowed = reproduce_ge19_windowed()

for i in windowed.instances:
    key, pin = f"n{i.n}", GE19_WINDOWED_ACHIEVED[f"n{i.n}"]
    assert (
        i.total_ccz,
        i.adder_ccz,
        i.lookup_ccz,
        i.unlookup_ccz,
        i.bridged_ccz,
    ) == (
        pin["total_ccz"],
        pin["adder_ccz"],
        pin["lookup_ccz"],
        pin["unlookup_ccz"],
        pin["bridged_ccz"],
    ), f"n={i.n}: counts moved off their pins"
    assert i.adder_ccz + i.lookup_ccz + i.unlookup_ccz == i.total_ccz, (
        f"n={i.n}: terms do not sum to total"
    )
    assert (
        GE19_WINDOWED_TOL["table1_lo"][key]
        <= i.table1_ratio
        <= GE19_WINDOWED_TOL["table1_hi"][key]
    ), f"n={i.n}: Table 1 ratio {i.table1_ratio:.4f} outside declared band"
    assert (
        GE19_WINDOWED_TOL["bridged_table1_lo"][key]
        <= i.bridged_table1_ratio
        <= GE19_WINDOWED_TOL["bridged_table1_hi"][key]
    ), (
        f"n={i.n}: bridged Table 1 ratio {i.bridged_table1_ratio:.4f} outside declared band"
    )
    assert within(
        i.total_ccz, i.closed_form_16, GE19_WINDOWED_TOL["rel_closed_form_16"]
    ), f"n={i.n}: total {i.total_ccz:.4e} off 16*ne*n^2/lg^2n"

table(
    ["n", "n_e", "g_pad", "window", "total n_ccz", "adder", "lookup", "unlookup"],
    [
        [
            i.n,
            i.exp_bitsize,
            i.coset_padding,
            f"({i.exp_window},{i.mul_window})",
            f"{i.total_ccz:,}",
            f"{i.adder_ccz / i.total_ccz:.2%}",
            f"{i.lookup_ccz / i.total_ccz:.2%}",
            f"{i.unlookup_ccz / i.total_ccz:.2%}",
        ]
        for i in windowed.instances
    ],
)
print()
table(
    ["n", "measured", "/T1", "bridged", "bridged/T1", "/24 form", "/16 form"],
    [
        [
            i.n,
            f"{i.total_ccz:.4e}",
            f"{i.table1_ratio:.4f}",
            f"{i.bridged_ccz:.4e}",
            f"{i.bridged_table1_ratio:.4f}",
            f"{i.total_ccz / i.closed_form_24:.4f}",
            f"{i.total_ccz / i.closed_form_16:.4f}",
        ]
        for i in windowed.instances
    ],
)

### Regime identification

`total/(n_e*n^2)` over n in {128 ... 8192}, which must fall like `1/lg^2 n`. Section 2 ran
the same measurement on `ModExp` and got a constant, so the two regimes separate on scaling
alone, with no external number.

In [ ]:
ns, cs = zip(*windowed.coefficient_series)
fall = cs[0] / cs[-1]
lg_rise = math.log2(ns[-1]) ** 2 / math.log2(ns[0]) ** 2
scaled = dict(windowed.coefficient_series)[N] * math.log2(N) ** 2
modexp_spread = coeffs[0] / coeffs[-1]

assert monotone(list(cs), increasing=False, strict=True), (
    "windowed coefficient does not fall at every step"
)
assert lg_rise / fall > 0.5, (
    f"1/lg^2n accounts for only {lg_rise:.2f}x of the {fall:.2f}x fall"
)
assert (
    GE19_WINDOWED_TOL["falloff_lg2_lo"] <= scaled <= GE19_WINDOWED_TOL["falloff_lg2_hi"]
), f"coeff(n={N}) * lg^2n = {scaled:.4f} outside declared band"
assert fall > 2 and modexp_spread < 1.02, (
    f"regimes not separated: windowed {fall:.2f}x vs ModExp {modexp_spread:.3f}x"
)

table(
    ["n", "coefficient", "x lg^2n"],
    [
        [n, f"{c:.6f}", f"{c * math.log2(n) ** 2:.4f}"]
        for n, c in windowed.coefficient_series
    ],
)

### Window grid

The full `(w_e, w_m)` grid at n=2048, then the per-n cost argmin beside GE19's published
`(5, 5)` at each tabulated size.

`(5, 5)` comes from GE19 L690; the grid minimum checks that value rather than justifying
it. The per-n window is the cost argmin: at n=1024 that lands further from Table 1 than
`(5, 5)` would, so both are printed.

In [ ]:
grid = sorted(windowed.window_grid, key=lambda r: r[2])

assert windowed.window_argmin == DEFAULT_WINDOW, (
    f"grid minimum {windowed.window_argmin} is not GE19's published {DEFAULT_WINDOW}"
)
assert all(
    i.table1_ratio <= windowed_total_ccz(i.n, *DEFAULT_WINDOW) / i.table1_toffoli
    for i in windowed.instances
), "cost argmin lands closer to Table 1 than (5,5) at some n"
assert grid[-1][2] / grid[0][2] > 5, (
    "window grid is flat; window choice would not matter"
)

table(
    ["w_e", "w_m", "k", "2^k", "total n_ccz", "/T1"],
    [
        [
            we,
            wm,
            we + wm,
            f"{2 ** (we + wm):,}",
            f"{t:,}",
            f"{t / (TABLE1['n2048'] * 1e9):.4f}",
        ]
        for we, wm, t in grid
    ],
)
print()

table(
    [
        "n",
        "argmin",
        "argmin n_ccz",
        "argmin/T1",
        "(5,5) n_ccz",
        "(5,5)/T1",
        "argmin vs T1",
    ],
    [
        [
            i.n,
            f"({i.exp_window},{i.mul_window})",
            f"{i.total_ccz:,}",
            f"{i.table1_ratio:.4f}",
            f"{windowed_total_ccz(i.n, *DEFAULT_WINDOW):,}",
            f"{windowed_total_ccz(i.n, *DEFAULT_WINDOW) / i.table1_toffoli:.4f}",
            "further"
            if i.table1_ratio
            < windowed_total_ccz(i.n, *DEFAULT_WINDOW) / i.table1_toffoli
            else "=",
        ]
        for i in windowed.instances
    ],
)

## 4. GE19 physical

GE19's formula count, not the `ModExp` count, through the CCZ2T grid search at
`phys_err = 1e-3`, `cycle_time_us = 1.0`, `n_algo_qubits = 6189`. Both rows use the same
search and differ only in `n_factories`.

Both free parameters are GE19-published: `error_budget = 0.31` is the paper's retry risk
(Table 3; L1086 defines it as "the overall probability of errors occurring", which is what
Qualtran's `error_budget` means), and `n_factories = 28` is Table 2's factory count for the
parallel row.

Conventions do not mix. qrepro emits a per-run duration and has no retry model; GE19
Table 2 quotes expected, Table 3 quotes per run, and L1096 gives the conversion
$t/(1-\epsilon)$. Every row below is labelled with the comparison it makes (sec. 3).

In [ ]:
physical = reproduce_ge19_physical()
t3 = GE19["physical_rows"]["table3_authoritative"]

for r in physical.rows:
    if r.deviation is None:
        continue
    tol = GE19_TOL["rel_qubits" if "qubits" in r.metric else "rel_runtime"]
    assert abs(r.deviation) <= tol, (
        f"{r.label}/{r.metric}: {r.deviation:+.1%} exceeds {tol:.0%}"
    )
assert (physical.parallel.factory_l1_d, physical.parallel.factory_l2_d) == (
    t3["d1"],
    t3["d2"],
), "grid search does not select GE19's own factory"
assert within(
    t3["runtime_hr_per_run"] / (1 - t3["retry"]) / 24,
    GE19["physical_rows"]["parallel"]["runtime_days"],
    0.01,
), "GE19 Tables 2 and 3 do not reconcile via the published retry risk"

table(
    ["row", "metric", "qrepro", "GE19", "dev"],
    [
        [
            r.label,
            r.metric,
            f"{r.reproduced:.5g}",
            "" if r.target is None else f"{r.target:.4g}",
            "" if r.deviation is None else f"{r.deviation:+.1%}",
        ]
        for r in physical.rows
    ],
)

Sensitivity of the reproduction to `error_budget` and `n_factories`, the two GE19-published
free parameters.

The 1-factory runtime comparison cannot be aligned: GE19 publishes only an expected runtime
for that scenario and Table 3's per-run figure covers the n=2048 optimum only, so there is
no per-run target for that layout. Both readings are printed; resolving it needs a retry
risk GE19 does not publish.

In [ ]:
retry = t3["retry"]
per_run = physical.one_factory_runtime_hr
target = physical.one_factory_target_runtime_hr_expected
par = [p for _, _, p in physical.sweep]

assert (
    max(p.physical_qubits for p in par) / min(p.physical_qubits for p in par) < 1.3
), "parallel row is sensitive to the error budget"
assert all(c.budget_satisfied for _, c in physical.factory_sweep), (
    "some swept configuration does not satisfy its budget"
)
assert (per_run < target) != (per_run / (1 - retry) < target), (
    "the two runtime conventions do not bracket the target"
)

table(
    ["eb", "1f qubits", "1f hr/run", "1f d", "28f qubits", "28f hr/run", "28f d"],
    [
        [
            f"{eb:.2f}",
            f"{o.physical_qubits / 1e6:.2f}M",
            f"{o.wall_time_us / 3.6e9:.2f}",
            o.code_distance,
            f"{p.physical_qubits / 1e6:.2f}M",
            f"{p.wall_time_us / 3.6e9:.2f}",
            p.code_distance,
        ]
        for eb, o, p in physical.sweep
    ],
)
print()
table(
    ["nf", "qubits", "hr/run", "d_data", "d1", "d2", "fail"],
    [
        [
            nf,
            f"{c.physical_qubits / 1e6:.2f}M",
            f"{c.wall_time_us / 3.6e9:.2f}",
            c.code_distance,
            c.factory_l1_d,
            c.factory_l2_d,
            f"{c.failure_prob:.3f}",
        ]
        for nf, c in physical.factory_sweep
    ],
)

## 5. 2019 to 2025 decomposition

G2025's logical counts (Table 5: 6.5e9 Toffolis on 1399 logical qubits at n=2048) through
the same 2019-era CCZ2T model as GE19 at the same factory count, so no layout difference
enters the algorithmic share.

The Toffoli conventions differ and are normalised first. GE19 Table 1 is per run (L1788:
"does not account for the chance of retrying"); G2025 Table 5 is expected per factoring,
already aggregating $E(\text{shots}) = 9.2$. Physical qubits are a per-shot resource, so
feeding both in raw compares different quantities. Both normalisations are computed.

This is a decomposition, not a reproduction: G2025's published < 1e6 rests on yoked surface
codes and magic-state cultivation, which have no representation in any open cost model
checked (Qualtran, Azure QRE, pyLIQTR). The residual gap is reported, not closed.

In [ ]:
ratios, ge19_qubits, g2025_qubits, blocks = [], [], [], []
for convention in CONVENTIONS:
    decomp = reproduce_2019_to_2025(convention)
    ge19_t, g2025_t = _toffoli_counts(convention)
    ratios += [r.algorithmic_ratio for r in decomp.factory_rows]
    ge19_qubits += [r.ge19.physical_qubits / 1e6 for r in decomp.factory_rows]
    g2025_qubits += [r.g2025.physical_qubits / 1e6 for r in decomp.factory_rows]
    blocks.append((convention, ge19_t, g2025_t, decomp.factory_rows))

spread_conv = max(ratios[i] - ratios[i + 3] for i in range(3))
spread_nf = max(max(ratios[:3]) - min(ratios[:3]), max(ratios[3:]) - min(ratios[3:]))

assert (
    G2025_TOL["algo_ratio_lo"] <= min(ratios)
    and max(ratios) <= G2025_TOL["algo_ratio_hi"]
), f"ratio range {min(ratios):.2f}x - {max(ratios):.2f}x outside declared band"
assert spread_nf > spread_conv, (
    f"convention spreads the range more than factory count ({spread_conv:.2f} vs {spread_nf:.2f})"
)
assert min(g2025_qubits) > 1.0, (
    f"model reaches G2025's sub-million target (floor {min(g2025_qubits):.2f} M)"
)

for convention, ge19_t, g2025_t, rows in blocks:
    print(
        f"\nconvention={convention}  (GE19 {ge19_t:.4e} Toffoli, G2025 {g2025_t:.4e})"
    )
    table(
        ["factories", "GE19 qubits", "G2025 qubits", "algorithmic x"],
        [
            [
                r.n_factories,
                f"{r.ge19.physical_qubits / 1e6:.2f}M",
                f"{r.g2025.physical_qubits / 1e6:.2f}M",
                f"{r.algorithmic_ratio:.2f}x",
            ]
            for r in rows
        ],
    )